# 🚀 Automatische Erkennung von Einstellungen mit *TransNetV2*

Dieses Notebook annotiert die Abfolge und die Länge der Einstellungen innerhalb einer oder mehrerer Videodateien mithilfe der **Shot Boundary Detection**, also der automatisierten Erkennung von Übergängen zwischen Einstellungen. Als Ergebnis entstehen:

- eine Tabelle im `.csv`-Format mit Start, Ende, Mittelpunkt und Dauer jeder erkannten Einstellung,
- eine weitere Tabelle im `.csv`-Format zu den Metadaten der Analyse
- eine ELAN-Datei (`.eaf`) zum Import und zur Sichtung der Annotationsdaten in *VIAN* oder anderen Programmen zur Videoannotation.

Beim erneuten Export werden weitere Versionen der Ergebnisdateien angelegt. Bereits vorhandene Dateien bleiben erhalten.

Der Arbeitsablauf ist in vier Abschnitte gegliedert: **Festlegen der Parameter**, **technische Vorbereitung**, **Analyse der Einstellungsübergänge** und **Export der Ergebnisse**. Die Texte zwischen den Codeblöcken erläutern vor allem den Zweck dieser einzelnen Schritte und die Bedeutung der erzeugten Dateien, während die Kommentare innerhalb des Codes auf Englisch dessen Funktionsweise vermitteln.

Weitere Informationen zur Funktionsweise dieses Notebooks bietet das <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/einstellung_intro.html" class="external-link" target="_blank">Einführungskapitel zur Einstellungserkennung innerhalb der OER</a>.

## Ausführung des Notebooks
Im Jupyter Book, also als Seite innerhalb der OER, ist das Ergebnis der Ausführung gespeichert und kann nachvollzogen werden, ohne den Code selbst auszuführen.
Wir empfehlen allerdings für ein tieferes Verständnis und die Bearbeitung der späteren Übungen, das Notebook in einer Cloud Umgebung oder lokal auszuführen. 

**Cloud Umgebung**: Klicken Sie oben rechts in der Menüleiste auf das Raketen-Symbol 🚀 und wählen Sie „Colab“, um das Notebook auszuführen. Denken Sie daran, den <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/technik.html#ausfuhrungsmodi" class="external-link" target="_blank">Laufzeittyp zu ändern</a>, um das Notebook ggf. mit einer GPU auszuführen.

**Lokale Umgebung**: Klicken Sie oben rechts in der Menüleiste auf das Download-Symbol ↓ und wählen Sie „.ipynb“, um das Notebook herunterzuladen und lokal auszuführen. Abhängig von Browser und Betriebssystem müssen Sie die Datei mit Rechtsklick auf „.ipynb“ ggf. manuell herunterladen.

Wir empfehlen die Codeblöcke einzeln nacheinander auszuführen und nicht mit Funktionen wie "Alle ausführen" in Colab oder "Run All Cells" in anderen Anwendungen zur Ausführung von Jupyter Notebooks. Die separate Ausführung der Codeblöcke erleichtert das Verständnis der mit ihnen verbundenen Funktionen und gewährleistet, dass alle vorbereitenden Schritte für nachfolgende Codeblöcke bereits durchlaufen wurden.



## Festlegen der Parameter

In diesem Abschnitt werden die zentralen Parameter für die spätere Analyse festgelegt. Sie können an dieser Stelle angepasst werden, ohne Änderungen im weiteren Code vornehmen zu müssen. Zum Erlernen der Funktionen des Codes in diesem Notebook und für einen ersten Probelauf empfiehlt es sich, die voreingestellten Werte beizubehalten.

### Ordnerpfade

Die Variable `VIDEO_PATH` benennt den Ordner, innerhalb dessen die zu analysierenden Videodateien abgelegt werden. Falls dieser noch nicht existieren sollte, wird ein Ordner mit entsprechender Benennung angelegt. Die einzelnen Videos können dabei sowohl in Unterordnern, als auch auf der höchsten Ebene des Ordners abgelegt werden:

#### Option 1: Videodateien direkt im Videoordner

```text
VIDEO_PATH
├── video_01.mp4
├── video_02.mp4
└── video_03.mp4
```

Alle zu analysierenden Videodateien befinden sich direkt auf der höchsten Ebene des durch `VIDEO_PATH` festgelegten Ordners.

#### Option 2: Videodateien in Unterordnern

```text
VIDEO_PATH
├── film_01/
│   └── video_01.mp4
├── film_02/
│   └── video_02.mp4
└── film_03/
    └── video_03.mp4
```

Die zu analysierenden Videodateien befinden sich in einzelnen Unterordnern innerhalb des durch `VIDEO_PATH` festgelegten Ordners.

Eine Kombination beider Ablageoptionen ist ebenfalls möglich.

**Wichtig** ist, dass jede Videodatei einen einzigartigen Namen tragen muss. Dies gilt auch für Videodateien, die sich in unterschiedlichen Unterordnern befinden. Da das Notebook die Dateinamen ohne ihre Endungen zur Identifikation verwendet, dürfen beispielsweise nicht gleichzeitig `film.mp4` und `film.mov` vorhanden sein. In diesem Fall gibt das Notebook eine Fehlermeldung aus und unterbricht die Erfassung, bis die Dateinamen entsprechend geändert wurden.


In diesen Ordner werden auch die für diese *OER* vorbereiteten Testdateien im `.zip`-Format abgelegt. Diese sind für die Ausführung dieses Jupyter Notebooks zu Lernzwecken vorgesehen und wir empfehlen, diese hierfür auch zu verwenden. Ist deren Download nicht erwünscht, kann die Variable `DOWNLOAD_TESTFILES` auf `False` gesetzt werden. In diesem Fall müssen eigene Videodateien in den Ordner kopiert werden, der unter dem Parameter `VIDEO_PATH`angegeben ist.

Zusätzlich können auch eigene `.zip`-Archive von Online-Speicherplätzen heruntergeladen werden. Hierzu muss ein entsprechender Download-Link innerhalb von Anführungszeichen dem Parameter `CUSTOM_ZIP_LINK` zugeordnet werden, z.B. `CUSTOM_ZIP_LINK = "https://example.org/video_archive.zip"`. Wichtig ist dabei, dass es sich um einen Direktlink zur `.zip`-Datei handelt, nicht um einen Link zu einer Website. Außerdem sollte beim Download mehrerer Archive nacheinander jeweils die Variable `CUSTOM_ARCHIVE_NAME` geändert werden, da Archive mit demselben Namen nicht erneut heruntergeladen werden. Die Option zum Download eigener Dateien ist besonders bei der Analyse von Langfilmen in *Google Colab* hilfreich, da die Videodateien dadurch nicht manuell und unter erheblichem Zeitaufwand in die *Colab*-Umgebung hochgeladen werden müssen. Der Download von einem Online-Speicher wie z.B. einem Cloud-Dienst benötigt in der Regel weniger Zeit.

Der Parameter `EXPORT_PATH` bezeichnet den Pfad zum Exportordner.

### Festlegung des Schwellenwerts

Besonders wichtig ist der Schwellenwert `THRESHOLD`. Er bestimmt, ab welcher vom Modell berechneten Wahrscheinlichkeit ein Bildwechsel als Einstellungsgrenze gewertet und als Annotation in die Ausgabe der Ergebnisse aufgenommen wird. Ein höherer Wert führt tendenziell zu weniger, aber sichereren Treffern. Ein niedrigerer Wert führt zur Erkennung von mehr Übergängen, führt allerdings auch zu einer höheren Wahrscheinlichkeit falsch-positiver Ergebnisse ("*false positives*"). Der Werteumfang für die Wahrscheinlichkeit liegt dabei zwischen `0.0` für 0% und `1.0` für 100%. Eine ausführlichere Einführung zu Schwellenwerten finden Sie im <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/vision/begriffe.html" class="external-link" target="_blank">OER-Abschnitt zu zentralen Begriffen und Konzepten</a>.



In [1]:
# Path to the directory containing the video files to be analysed.
# The default relative path is resolved from the location where the notebook is executed.
VIDEO_PATH = "videos_shots"

# The video directory is created automatically if it does not exist yet.
# `exist_ok=True` prevents an error if the directory already exists.
import os
os.makedirs(VIDEO_PATH, exist_ok=True)

# Path to the directory for the export of the analysis results
EXPORT_PATH = "export_shots"
os.makedirs(EXPORT_PATH, exist_ok=True)

# Minimum probability at which TransNetV2 treats a frame as a confirmed shot boundary.
# The value ranges from 0 to 1.
THRESHOLD = 0.5

# If True, a provided test video is downloaded and extracted.
# Set this option to False when analyzing your own files.
DOWNLOAD_TESTFILES = True

# Add your own custom link inside quotes to download additional files into the video folder.
CUSTOM_ZIP_LINK = "insert link here"
# Change the name of the custom archives. A new archive with the same name will not be downloaded. 
CUSTOM_ARCHIVE_NAME = "custom_archive"

# Separator line for visually clearer text output.
DIVIDER = "─" * 100

---
## Technische Vorbereitung

Bevor ein Video analysiert werden kann, müssen die benötigten Softwarepakete installiert, in das Notebook eingebunden und das vortrainierte TransNetV2-Modell geladen werden.

### Installation der Programmbibliotheken

Der folgende Codeblock installiert externe Programmbibliotheken, die nicht in der Standardinstallation von *Python* enthalten sind. Dazu zählen neben dem Modell `TransNetV2` zur Erkennung der Einstellungsübergänge
- `pandas` zum Bearbeiten und Exportieren von tabellarischen Daten,
- `opencv-python` und `ffmpeg` zur Videobearbeitung,
- und `pympi-ling` zum Schreiben von *ELAN*-Dateien (`.eaf`).

Für die wichtigsten Bibliotheken wird eine spezifische Version installiert, um deren interdependente Funktionsweise zu garantieren.

In einer lokalen und bei einer erneuten Ausführung des Notebooks identischen Arbeitsumgebung muss diese Zelle nicht bei jeder Sitzung erneut ausgeführt werden. In *Google Colab* ist eine erneute Installation nach dem Neustart der Laufzeit hingegen notwendig, da alle speziell installierten Programmbibliotheken beim Beenden der Laufzeit gelöscht werden.


In [2]:
print("Installing dependencies.")

# "-q" stands for "quiet" and reduces the amount of installation output.

# The pretrained model and its PyTorch implementation.
!pip install -q transnetv2-pytorch==1.0.5

# Table processing and .csv file export.
!pip install -q pandas

# Reading video files and accessing technical video metadata.
!pip install -q opencv-python==5.0.0.93
!pip install -q static-ffmpeg==3.0

# Creating ELAN-annotation files in the .eaf-format.
!pip install -q pympi-ling
print("Dependencies installed.")

Installing dependencies.
Dependencies installed.


### Import der Programmbibliotheken

Durch den Import werden die installierten Funktionen nun in der aktuellen Notebook-Sitzung verfügbar. Die Bibliotheken übernehmen unterschiedliche Aufgaben, die im Codeblock jeweils kurz erläutert werden.


In [3]:
# Pretrained neural network for shot-boundary detection.
from transnetv2_pytorch import TransNetV2

# Running and managing the model
import torch

# Handling file and directory paths
from pathlib import Path

# Editing and Exporting tabular data
import pandas as pd

# Writing ELAN files.
import pympi

# Representing and calculating time values.
import datetime
import time

# Reading videos and their technical properties.
import cv2
import static_ffmpeg

# Downloading a file from a URL and extracting ZIP archives.
from urllib.request import urlretrieve
import zipfile

print("Dependencies imported.")

Dependencies imported.


### Download des Analysematerials
Zum Download des Testmaterials und eines eigenen `.zip`-Archivs wird eine wiederverwendbare Funktion definiert und direkt angewendet. 

Falls bei der [Festlegung der Parameter](#festlegen-der-parameter) `DOWNLOAD_TESTFILES = True` gesetzt ist, lädt das Notebook das `.zip`-Archiv `shot_detection_testfiles.zip` herunter, das den Trailer zu Konrad Wolfs Film *Der geteilte Himmel* enthält. Liegt `shot_detection_testfiles.zip` aufgrund einer vorherigen Ausführung bereits auf derselben Ordnerebene wie das Notebook, wird die Datei nicht erneut heruntergeladen. Anschließend wird ihr Inhalt in den zuvor festgelegten Videoordner entpackt. Zu Lernzwecken im Rahmen der OER empfehlen wir den Download und die Arbeit mit den Testdateien im Notebook. Herkunft und Zusammensetzung des Testarchivs erläutert der OER-Abschnitt <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/einstellung_intro.html#analysematerial" class="external-link" target="_blank">zum Analysematerial</a>.

Dieser Schritt ist für die Arbeit mit eigenen Video- und Film-Korpora nicht erforderlich. Eigene Videodateien können direkt im `VIDEO_PATH` Ordner oder in dessen Unterordnern abgelegt werden. In diesem Fall kann `DOWNLOAD_TESTFILES = False` gesetzt werden. Außerdem kann mittels des Parameters `CUSTOM_ZIP_LINK` ein eigenes `.zip`-Archiv heruntergeladen werden, wie bereits eingangs im Kontext der [Festlegung der Parameter](#festlegen-der-parameter) erläutert. In beiden Fällen sollte der Codeblock zur [Festlegung der Parameter](#festlegen-der-parameter) bereits ausgeführt worden sein, um falls notwendig den Videoordner erst zu erstellen.

In [4]:
# Define a reusable function for downloading .zip archives.
# This function takes three arguments: The download link, the name for the downloaded archive, and the directory to unzip the archive into.
def download_zip(url: str, archive_name: str, target_directory: str):

    # Add .zip to the filename and turn it into a relative path in the working directory.
    archive_path = Path(f"{archive_name}.zip")

    # Download only if the archive is not already present in the working directory.
    if not archive_path.is_file():
        # Download the file.
        print(f"Downloading {archive_path}...")
        urlretrieve(url, archive_path)
        print(f"{archive_name} downloaded.")
    else:
        print(f"{archive_name} already downloaded.")

    # Unzip the file to the target_directory.
    target_directory = Path(target_directory)
    with zipfile.ZipFile(archive_path,"r") as zip_ref:
        zip_ref.extractall(target_directory)
        print(f"Extracted {archive_name} to {target_directory}.")

# Download and extract the testfiles if the setting is enabled. 
if DOWNLOAD_TESTFILES == True:
    print("Testfile download requested.")
    
    # Download the testfiles
    url = ("https://owncloud.gwdg.de/index.php/s/FXBHduaTJcavU7l/download")
    download_zip(url, "shot_detection_testfiles", VIDEO_PATH)
else:
     print("No testfile download requested.")

# Download and extract the custom archive if a link is provided.
if CUSTOM_ZIP_LINK != "insert link here":
    print("Custom file download requested.")
 
    # Download the custom video archive.
    download_zip(CUSTOM_ZIP_LINK, CUSTOM_ARCHIVE_NAME, VIDEO_PATH)

Testfile download requested.
shot_detection_testfiles downloaded.
Extracted shot_detection_testfiles to videos_shots.


### Vorbereitung des `TransNetV2`-Modells

In diesem Schritt wird das *Machine-Learning-Modell* <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/einstellung_intro.html#funktionsweise-des-transnetv2-modells" class="external-link" target="_blank">TransNetV2</a> zur Erkennung der Einstellungsübergänge vorbereitet. Es handelt sich um ein bereits trainiertes neuronales Netz. Es berechnet für jeden Frame eines Videos einen Wahrscheinlichkeitswert, der angibt, wie wahrscheinlich es ist, dass an dieser Stelle ein Übergang zwischen zwei Einstellungen stattfindet.

Mit `model.eval()` wird das neuronale Netz in den Auswertungsmodus versetzt. Dieser Modus ist für die Anwendung eines trainierten Modells vorgesehen, nicht für dessen weiteres Training.

In [5]:
# Create an instance of the pretrained model.
model = TransNetV2()

# Activate evaluation mode: the model should make predictions rather than continue training.
model.eval()

# Display the compute device being used, usually the CPU or, if configured accordingly, the GPU.
print(f"Model loaded on: {next(model.parameters()).device}")

# Add the programs provided by "static-ffmpeg" to the search path.
# TransNetV2 requires ffmpeg/ffprobe to decode video files reliably.
static_ffmpeg.add_paths()

ℹ️  MPS device detected but not used due to numerical inconsistency issues.
   Use --device mps to explicitly enable MPS (faster but inconsistent results).
Model loaded on: cpu


True

---
## Analyse der Einstellungsübergänge

Hier beginnt die eigentliche Analyse. Zunächst werden die Videodateien erfasst und deren Metadaten gesammelt. Anschließend führt das Notebook mithilfe des Modells die eigentliche Einstellungserkennung durch und bereitet deren Ergebnis tabellarisch auf.

### Erfassung der technischen Videometadaten

Für jede Videodatei werden deren 
- Dateiname (`video_name`) und Ablageort (`path`),
- Filmtitel (`title`) und gegebenenfalls das aus dem Dateinamen ermittelte Erscheinungsjahr (`release_year`),
- Auflösung (`frame_height` und `frame_width`),
- Anzahl der Frames (`frame_count`),
- Bildrate (`frame_rate`) und
- berechnete Laufzeit (`duration_hh_mm_ss`)

bestimmt. Diese Angaben sind sowohl für die Interpretation des Analyseergebnisses als auch für die spätere Umrechnung von Framepositionen in Sekunden notwendig.

Die Informationen aller Videos werden im **Dictionary** `videos` gesammelt. Ein **Dictionary** ist eine Python-Datenstruktur, in der jedem Schlüssel (`key`) ein Wert zugeordnet wird. Im vorliegenden Fall dient der Name eines Videos als Schlüssel. Der zugehörige Wert ist ein weiteres Dictionary, das den Dateipfad, technische Metadaten und später auch die Analyseergebnisse enthält. Weitere Informationen und Beispiele zur Funktionsweise eines Dictionaries finden Sie etwa im <a href="https://www.python-kurs.eu/python3_dictionaries.php" class="external-link" target="_blank">Dictionary-Kapitel des hier verlinkten Python Kurses</a>.

In [6]:
# Supported video file extensions.
VIDEO_EXTENSIONS = {
    ".mp4", ".avi", ".mov", ".mkv", ".wmv",
    ".flv", ".webm", ".mpeg", ".mpg", ".m4v", ".3gp"
}

# Store each video's path and technical metadata.
# Video filenames must be unique across all subdirectories.
videos = {}

# Recursively search the input directory for video files.
for video_path in Path(VIDEO_PATH).rglob("*"):

    # Ignore directories and other non-file entries.
    if not video_path.is_file():
        continue

    # Ignore files inside Jupyter checkpoint directories.
    if ".ipynb_checkpoints" in video_path.parts:
        continue

    # Ignore unsupported file formats.
    if video_path.suffix.lower() not in VIDEO_EXTENSIONS:
        continue

    # Use the filename without its extension as the dictionary key.
    video_name = video_path.stem
    # Extract the release year from the filename and check if it is an integer
    try:
        video_title, release_year = video_name.rsplit("_", 1)
        release_year = int(release_year)
        print(f"Successfully extracted video name {video_title} and release year {release_year}.")
    except:
        release_year = None
        video_title = video_name
        print(f"Filename {video_name} does not provide a release year. The correct formatting is 'title_year'.")

    if video_name in videos:
        raise ValueError(
            f"Duplicate video name: {video_name}. "
            "Please use unique filenames."
        )

    # Open the video without loading the complete file into memory.
    capture = cv2.VideoCapture(str(video_path))

    if not capture.isOpened():
        print(f"Video could not be opened: {video_path}")
        capture.release()
        continue

    # Read the video's technical metadata.
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_rate = capture.get(cv2.CAP_PROP_FPS)

    # Release the video file after reading its metadata.
    capture.release()

    # Skip files with invalid metadata.
    if frame_rate <= 0 or frame_count <= 0:
        print(f"Video contains invalid metadata: {video_path}")
        continue

    # Calculate the video duration and convert it to HH:MM:SS.
    duration_hh_mm_ss = str(
        datetime.timedelta(seconds=round(frame_count / frame_rate))
    )

    print(f"Video file: {video_name}")
    print(
        f"Resolution: {frame_width}x{frame_height} | "
        f"Frames: {frame_count} | "
        f"Frame Rate: {frame_rate:.3f} | "
        f"Duration: {duration_hh_mm_ss}"
    )
    print(DIVIDER)

    # Initialise the video dictionary.
    videos[video_name] = {
        "path"              : video_path,
    }
    # Store the metadata for the subsequent analysis steps.
    videos[video_name]["metadata"]= {
        "video_name"        : video_path.name,
        "title"             : video_title,
        "release_year"      : release_year,
        "frame_count"       : frame_count,
        "frame_height"      : frame_height,
        "frame_width"       : frame_width,
        "frame_rate"        : frame_rate,
        "duration_hh_mm_ss"       : duration_hh_mm_ss,
        "model"                   : "transnetv2",
        "device"                  : next(model.parameters()).device.type,
        "analysis_timestamp"      : None,  # Updated when analysis begins.
        "processing_time_hh_mm_ss": None   # Updated after the analysis.
    }

Successfully extracted video name Der geteilte Himmel_Trailer and release year 1964.
Video file: Der geteilte Himmel_Trailer_1964
Resolution: 1920x1080 | Frames: 5225 | Frame Rate: 25.000 | Duration: 0:03:29
────────────────────────────────────────────────────────────────────────────────────────────────────


### Anwendung des Modells

Im folgenden Codeblock werden nacheinander die zuvor gesammelten Videos analysiert. 

Für jedes Video berechnet `TransNetV2` pro Frame die Wahrscheinlichkeit für einen Einstellungsübergang. Dabei entstehen drei Ergebnisse:

- `video_frames` ist eine Liste der extrahierten Frames, die etwa zur Erstellung einer Visualisierung des Ergebnisses genutzt werden können.

- `all_frame_predictions` enthält die vom Modell berechneten Wahrscheinlichkeiten dafür, dass ein Frame Teil eines Einstellungsübergangs ist. Bei schrittweisen Übergängen, beispielsweise Überblendungen oder Auf- und Abblenden, können somit mehrere aufeinanderfolgende Frames hohe Wahrscheinlichkeiten erhalten.

- `single_frame_predictions` enthält hingegen die Wahrscheinlichkeiten dafür, dass ein Frame die Grenze zwischen zwei Einstellungen markiert. Das Modell wurde darauf trainiert, auch bei längeren Übergängen einen einzelnen Frame, in der Regel in der Mitte des Übergangs, als Einstellungsgrenze zu identifizieren. Die folgenden Schritte verwenden deshalb `single_frame_predictions` zur Bestimmung der Einstellungsgrenzen.

Die Funktion `predictions_to_scenes` übersetzt die Wahrscheinlichkeiten anschließend mithilfe des gewählten Schwellenwerts `THRESHOLD` in konkrete Einstellungssegmente (`shots`). Jedes Segment wird als Paar der Framepositionen `startframe` und `endframe` gespeichert. Der Schwellenwert `THRESHOLD` wird im Codeblock zur [Festlegung der Parameter](#festlegen-der-parameter) eingestellt.

Zusätzlich misst das Notebook die benötigte Rechenzeit für die Durchführung der Analyse (`elapsed_time_hh_mm_ss`).

Dieser Schritt ist sehr rechenintensiv! Je nach Videomaterial kann er von einer halben Stunde bis zu mehreren Stunden in Anspruch nehmen. 

In [7]:
# Destination directory for later export files.
export_directory = Path(EXPORT_PATH)

# Analyze each previously registered video in sequence.
for video_name, video in videos.items():
   # Record the start of this analysis in UTC and start the duration timer.
   video["metadata"]["analysis_timestamp"] = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
   start_time = time.perf_counter()

   # Calculate predictions without calculating gradients.
   # Gradients are only required when training models.
   # This reduces memory usage and speeds up computation.
   with torch.no_grad():
      video_frames, single_frame_predictions, all_frame_predictions = model.predict_video(video["path"])

      # Convert the PyTorch result to a NumPy array for further processing.
      prediction_scores = single_frame_predictions.cpu().detach().numpy()

      # Convert the probability scores into concrete shot segments.
      shots = model.predictions_to_scenes(prediction_scores, THRESHOLD)

      # Calculate the elapsed time from when the timer was started.
      elapsed_time_seconds = time.perf_counter() - start_time
      # Convert the elapsed time into the hhmmss format
      elapsed_time_hh_mm_ss = str(
         datetime.timedelta(seconds=round(elapsed_time_seconds))
         )

      print(f"Shot Boundary Detection on {video_name} took {elapsed_time_hh_mm_ss} of processing time.")
      print(DIVIDER)

      # Record the compute device used by the model.
      video["metadata"]["device"] = next(model.parameters()).device.type

      # Add the analysis result to the video dictionary
      video.update({
         "shots" : shots,
         "elapsed_time_hh_mm_ss" : elapsed_time_hh_mm_ss,
         "prediction_scores": prediction_scores
      })

Extracting frames from videos_shots/Der geteilte Himmel_Trailer_1964.mp4


Processing frames: 100%|██████████| 5225/5225 [02:24<00:00, 36.14frame/s]

Shot Boundary Detection on Der geteilte Himmel_Trailer_1964 took 0:02:35 of processing time.
────────────────────────────────────────────────────────────────────────────────────────────────────


### Tabellarische Aufbereitung des Ergebnisses

Die vom Modell gelieferten Einstellungen (`shots`) werden in einen `pandas`-DataFrame (`df_shots`) überführt, mittels dessen sich Datensätze einfacher bearbeiten und exportieren lassen. Ein Dataframe ist dabei ein spezielles Tabellenformat. Jede Tabellenzeile entspricht einer erkannten Einstellung. Mehr Informationen zu Dataframes finden Sie <a href="https://www.python-kurs.eu/pandas_DataFrame.php" class="external-link" target="_blank">in diesem Kapitel des zuvor verlinkten Python Kurses</a>.

Neben Startframe (`start`) und Endframe (`end`) werden der zeitliche Mittelpunkt (`middle`) und die Dauer (`duration`) jeder Einstellung berechnet. Die Zählung der Frames beginnt dabei bei `0`. Da der Endframe noch zur jeweiligen Einstellung gehört, wird bei der Berechnung der Dauer ein zusätzlicher Frame berücksichtigt (`end - start + 1`). Anschließend werden alle Frameangaben mithilfe der Bildrate (`frame_rate`) in Sekunden umgerechnet und auf zwei Nachkommastellen gerundet. Die Sekundenwerte erleichtern die weitere Auswertung der erkannten Einstellungen.


In [8]:
# Process the analysis results for each previously registered video in sequence.
for video_name, video in videos.items():
    # Build dataframe. Each row contains the start and end frame of a shot.
    df_shots = pd.DataFrame(video["shots"], columns=["start", "end"])

    # Calculate the middle frame of a shot.
    df_shots["middle"] = df_shots["start"] + ((df_shots["end"] - df_shots["start"]) / 2).astype(int)

    # Calculate the duration of the shot in number of frames
    df_shots["duration"] = df_shots["end"] - df_shots["start"] + 1

    # Convert frames to seconds.
    # Time in seconds = frame number divided by frames per second ("frame_rate") of the respective video.
    for col in ["start", "end", "middle", "duration"]:
        df_shots[f"{col}_s"] = round(df_shots[col] / video["metadata"]["frame_rate"], 2)

    # Preview the first five detected shots.
    print(f"First five shots in {video_name}:")
    print(df_shots.head(5))
    print(DIVIDER)

    # Add the complete table to the corresponding video dictionary
    video.update({
        "df_shots" : df_shots
    })

First five shots in Der geteilte Himmel_Trailer_1964:
   start   end  middle  duration  start_s  end_s  middle_s  duration_s
0      0   179      89       180     0.00   7.16      3.56        7.20
1    180   447     313       268     7.20  17.88     12.52       10.72
2    448   788     618       341    17.92  31.52     24.72       13.64
3    789   971     880       183    31.56  38.84     35.20        7.32
4    972  1186    1079       215    38.88  47.44     43.16        8.60
────────────────────────────────────────────────────────────────────────────────────────────────────


### Berechnung der Analysemetadaten

Neben den Ergebnissen der Einstellungserkennung erzeugt das Notebook pro Video eine weitere Tabelle `metadata`, in der die Metadaten zur Analyse eingetragen werden: 

- der angewendete Schwellenwert, 
- die Anzahl der Einstellungen, 
- der Mittelwert der Einstellungslängen, 
- der Median der Einstellungslängen, 
- die Dauer der längsten und der kürzesten Einstellung, 
- die Gesamtdauer des Videos, die für die Analyse benötigte Rechenzeit, 
- der minimale und der maximale Wahrscheinlichkeitswert bei der Erkennung, 
- die Auflösung und die Bildrate der Videodatei,
- das verwendete Modell (`model`) und das Rechengerät (`device`),
- sowie den Zeitpunkt des Analysebeginns in UTC (`analysis_timestamp`).

Eine vollständige Übersicht der exportierten Felder bietet die Metadatentabelle im OER-Abschnitt <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/einstellung_intro.html#ausgabe" class="external-link" target="_blank">zur Ausgabe des Notebooks</a>.

Diese Daten sind nützlich, um eine kompakte Übersicht über das Ergebnis zu erhalten oder um etwa die Auswirkungen unterschiedlicher `THRESHOLD`-Werte auf die Erkennungsergebnisse zu vergleichen. Außerdem werden diese Daten für die <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/auswertung/visualisierung_einstellungen.html" class="external-link" target="_blank">Visualisierung der Einstellungsmetadaten im nächsten Kapitel</a> verwendet. 

Mittelwert (`mean`) und Median (`median`) sind unterschiedliche Methoden zur Berechnung von Durchschnittswerten. Der Mittelwert berechnet sich aus der Summe der erkannten Einstellungsdauern geteilt durch deren Anzahl. Der Mittelwert reagiert stärker auf sehr lange Einstellungen. Der Median beschreibt hingegen die Dauer der Einstellung die genau in der Mitte liegt, wenn alle Einstellungen ihrer Länge nach nacheinander angeordnet werden (von der längsten bis zur kürzesten) und ist gegenüber Ausreißern robuster.

Die minimalen und maximalen Vorhersagewerte (`min_prediction_score` und `max_prediction_score`) geben den niedrigsten und höchsten vom Modell berechneten Wahrscheinlichkeitswert für einen Einstellungsübergang im untersuchten Video an. Diese Werte sind allerdings kein Maß für die Genauigkeit der Erkennung. Auch ein hoher Wahrscheinlichkeitswert bedeutet nicht zwangsläufig, dass ein Einstellungsübergang korrekt erkannt wurde. Dafür müssten die Ergebnisse mit manuell überprüften Annotationen verglichen werden.


In [9]:
# Calculate the metadata for each previously registered video in sequence.
for video_name, video in videos.items():
    video["metadata"].update({
        # Record the threshold used to ensure reproducibility.
        "threshold"                 : THRESHOLD,

        # Number of shots detected by the model.
        "shot_count"                : len(video["df_shots"]),

        # Descriptive statistics for shot duration in seconds.
        "mean_duration_s"           : round(video["df_shots"]["duration_s"].mean(), 2),
        "median_duration_s"         : round(video["df_shots"]["duration_s"].median(), 2),
        "max_duration_s"            : round(video["df_shots"]["duration_s"].max(), 2),
        "min_duration_s"            : round(video["df_shots"]["duration_s"].min(), 2),

        # Duration of the source video and time required for automatic processing.
        "duration_hh_mm_ss"          : video["metadata"]["duration_hh_mm_ss"],
        "processing_time_hh_mm_ss"  : video["elapsed_time_hh_mm_ss"],

        # Range of probability values produced by the model.
        "min_prediction_score"      : round(float(video["prediction_scores"].min()),2),
        "max_prediction_score"      : round(float(video["prediction_scores"].max()),2),            
    })

    # Transform the metadata dictionary into a dataframe.
    video["metadata"] = pd.DataFrame.from_dict(
        video["metadata"], orient="index", columns=["Value"]
    )
    print(f"Metadata for {video_name}:")
    display(video["metadata"])
    print(DIVIDER)

Metadata for Der geteilte Himmel_Trailer_1964:


,Value
video_name,Der geteilte Himmel_Trailer_1964.mp4
title,Der geteilte Himmel_Trailer
release_year,1964
frame_count,5225
frame_height,1080
frame_width,1920
frame_rate,25.0
duration_hh_mm_ss,0:03:29
model,transnetv2
device,cpu


────────────────────────────────────────────────────────────────────────────────────────────────────


---
## Export der Ergebnisse

Die Analyseergebnisse werden als nächstes in mehreren Formaten gespeichert. `.csv`-Dateien eignen sich für Tabellenkalkulation, Statistikprogramme und weitere Python-Auswertungen, etwa zur Visualisierung. Zudem können sie auch  in *OpenRefine* eingelesen werden. Die Verwendung von *OpenRefine* wird in der OER zu <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-2/bereinigung/openRefine/1_einf%C3%BChrung.html" class="external-link" target="_blank">OER zu Studentischen Filmen an der Filmuniversität</a> genauer erklärt, ebenso das <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-2/bereinigung/openRefine/2_import.html#datenformate" class="external-link" target="_blank">csv-Dateiformat</a>.

Das `.eaf`-Format kann direkt in *VIAN* oder anderen Programmen zur Videoannotation geöffnet oder importiert werden, die Daten können dann dort manuell überprüft, bearbeitet und ergänzt werden.

Beim erneuten Export werden zusätzliche Versionen der Ergebnisdateien erstellt. Vorhandene Dateien bleiben erhalten.


### Export der Tabellen als `.csv`-Dateien

Die Ergebnisse der Einstellungserkennung werden für jedes analysierte Video in zwei `.csv`-Dateien exportiert:

- Die Datei `…_shots_THRESHOLD.csv` enthält die einzelnen erkannten Einstellungen. Jede Zeile dokumentiert Beginn, Ende, Mittelpunkt und Dauer einer Einstellung, jeweils in Frames und Sekunden.
- Die Datei `…_shots_THRESHOLD_metadata.csv` enthält die technischen Metadaten des Videos, den verwendeten Analyseparameter `THRESHOLD` sowie die berechneten statistischen Ergebnisse der Einstellungserkennung. Die Angaben werden als Schlüssel-Wert-Paare in zwei Spalten gespeichert.

Weitere Informationen finden Sie im OER-Abschnitt <a href="https://quadriga-dk.github.io/Bewegtes-Bild-Fallstudie-4/annotation/einstellung_intro.html#ausgabe" class="external-link" target="_blank">zur Ausgabe des Notebooks</a>.

Beide Dateien werden direkt im zuvor festgelegten Exportordner (`EXPORT_PATH`) gespeichert. Anders als beim Notebook zur Figurenerkennung wird kein eigener Unterordner pro Video angelegt. Ihre Namen setzen sich aus dem Namen der Videodatei ohne Endung, `shots` und dem gewählten Schwellenwert (`THRESHOLD`) zusammen.

Die folgende Hilfsfunktion `next_export_paths()` prüft für jedes Video die beiden vorgesehenen CSV-Dateien und die zugehörige EAF-Datei gemeinsam. Beim ersten Export werden die ursprünglichen Dateinamen verwendet, beispielsweise `Filmtitel_shots_0.5.csv` und `Filmtitel_shots_0.5_metadata.csv`. Existiert bereits mindestens eine Datei dieses Satzes, wird für **alle Dateien des nächsten Exports** derselbe freie Versionszusatz vergeben (`_v2`, `_v3` usw.). So können bei einem zweiten Export `Filmtitel_shots_0.5_v2.csv`, `Filmtitel_shots_0.5_metadata_v2.csv` und `Filmtitel_shots_0.5_v2.eaf` entstehen. Auch eine unvollständige frühere Ausgabe reserviert ihre Versionsnummer.

Die ausgewählten Pfade werden unter `video["export_paths"]` gespeichert und beim EAF-Export wiederverwendet. Für eine neue vollständige Version werden zuerst die CSV-Exportzelle und danach die EAF-Exportzelle ausgeführt. Wird nur die EAF-Exportzelle erneut ausgeführt, bleibt eine bereits vorhandene EAF-Datei erhalten.


In [10]:
# Select one version for all export files belonging to the same video.
# Find a free version for the complete group of CSV and EAF files.
def next_export_paths(paths):
    version = 1
    while True:
        # The first export has no suffix; later versions use _v2, _v3, etc.
        suffix = "" if version == 1 else f"_v{version}"
        candidates = {
            kind: path.with_name(f"{path.stem}{suffix}{path.suffix}")
            for kind, path in paths.items()
        }
        # Even one existing file reserves this version for the whole group.
        if not any(path.exists() for path in candidates.values()):
            return candidates
        version += 1

# Export the tables for each previously registered video.
export_directory = Path(EXPORT_PATH)
export_directory.mkdir(parents=True, exist_ok=True)

for video_name, video in videos.items():
    # The threshold in the filename distinguishes different model settings.
    stem = f"{video_name}_shots_{THRESHOLD}"

    # Reserve one shared version for both CSV files and the later EAF file.
    video["export_paths"] = next_export_paths({
        "shots": export_directory / f"{stem}.csv",
        "metadata": export_directory / f"{stem}_metadata.csv",
        "eaf": export_directory / f"{stem}.eaf",
    })
    paths = video["export_paths"]

    # mode="x" refuses to overwrite a CSV file that already exists.
    video["df_shots"].to_csv(paths["shots"], index=False, mode="x")

    # Store the metadata as key-value pairs without a column heading.
    video["metadata"].to_csv(paths["metadata"], header=False, mode="x")

    print(f"Successfully exported {paths['shots']} & {paths['metadata']}.")
    print(DIVIDER)


Successfully exported export_shots/Der geteilte Himmel_Trailer_1964_shots_0.5.csv & export_shots/Der geteilte Himmel_Trailer_1964_shots_0.5_metadata.csv.
────────────────────────────────────────────────────────────────────────────────────────────────────


### Export der Einstellungsannotationen als `.eaf`-Datei

Die Datei `…_shots_THRESHOLD.eaf` enthält die erkannten Einstellungen als zeitliche Annotationen und wird ebenfalls direkt im Exportordner (`EXPORT_PATH`) gespeichert. Bei wiederholtem Export bleiben frühere Versionen der EAF-Datei erhalten.

Das `.eaf`-Format speichert Annotationen als Zeitangaben in Millisekunden. Deshalb werden die Start- und Endframes jeder Einstellung mithilfe der individuellen Bildrate des Videos direkt in Millisekunden umgerechnet. Der Endzeitpunkt liegt unmittelbar nach dem letzten zur Einstellung gehörenden Frame.

Für jedes Video wird eine Annotationsebene angelegt, die sich aus `shots` und dem gewählten Schwellenwert `THRESHOLD` zusammensetzt. Jede erkannte Einstellung erscheint dort als eigenes Zeitintervall. Die Intervalle enthalten anders als gewöhnlich keinen Text, sondern bilden eine zeitliche Segmentierung, die nach dem Import in VIAN dort auf ihre korrekte Erkennung hin überprüft und inhaltlich ergänzt werden kann.

In [11]:
# Export the .eaf annotations for each previously registered video in sequence.
for video_name, video in videos.items():
    # Reuse the version selected when exporting this video's CSV files.
    if "export_paths" not in video:
        raise RuntimeError("Run the CSV export cell before exporting EAF files.")
    eaf_document_path = video["export_paths"]["eaf"]

    # Prevent overwriting when the EAF export cell is run a second time.
    if eaf_document_path.exists():
        print(f"Skipping {eaf_document_path.name}: file already exists. Re-run the CSV export cell for a new version.")
        print(DIVIDER)
        continue

    frame_rate = float(video["metadata"].loc["frame_rate", "Value"])
  
    # Create a new, initially empty .eaf document.
    eaf_document = pympi.Eaf()
    # Add the annotation tier for the automatically detected shots.
    eaf_document.add_tier(f"shots_{THRESHOLD}")

    # Convert each table row into a time interval in milliseconds.
    for _, row in video["df_shots"].iterrows():
        eaf_document.add_annotation(
            f"shots_{THRESHOLD}",
            round(row["start"] / frame_rate * 1000),
            round((row["end"] + 1) / frame_rate * 1000)
        )

    # Write the EAF using the version already assigned to the CSV files.
    eaf_document.to_file(eaf_document_path)

    print(f"Successfully exported {eaf_document_path}.")
    print(DIVIDER)

Successfully exported export_shots/Der geteilte Himmel_Trailer_1964_shots_0.5.eaf.
────────────────────────────────────────────────────────────────────────────────────────────────────


### Herunterladen des *Google Colab* Export-Ordners

Dieser optionale Schritt ist nur für eine Ausführung in *Google Colab* vorgesehen. Der gesamte Exportordner in *Colab* wird zu einer `.zip`-Datei komprimiert und anschließend über die Browseroberfläche heruntergeladen. Der Download ist notwendig, da ansonsten die Erkennungsdaten verloren gehen, da diese nach Beenden der Laufzeit bzw. von *Colab* immer gelöscht werden, ebenso wie alle in Colab zur Analyse hochgeladenen Videodateien.

In einer lokalen *Jupyter Notebook*-Umgebung bleiben die Dateien im anfangs im Codeblock zur [Festlegung der Parameter](#festlegen-der-parameter) definierten Export-Ordner erhalten und können von dort entnommen werden. Es ist daher kein zusätzlicher Download-Schritt erforderlich.


In [12]:
try:
  # The following import works only within a Google Colab runtime.
  import google.colab
  from google.colab import files

  # Colab uses the `/content` directory by default.
  colab_export_directory = f"/content/{EXPORT_PATH}"
  zip_path = f"{EXPORT_PATH}.zip"
  # Package the entire results directory as a ZIP archive.
  !zip -r "{zip_path}" "{colab_export_directory}"

  # Start downloading the ZIP archive in the browser.
  files.download(f"/content/{EXPORT_PATH}.zip")
  print(f"Downloaded {EXPORT_PATH}.zip.")
except ImportError:
  # Skip this step when running locally.
  print("Download only works in Google Colab.")

Download only works in Google Colab.
